In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

DEFAULT_REPO_ROOT = Path(os.environ.get("CHESS_REPO_ROOT", "/content/drive/MyDrive/chess_engine"))
if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    if not DEFAULT_REPO_ROOT.exists():
        raise FileNotFoundError(
            f"Missing repo root: {DEFAULT_REPO_ROOT}. "
            "Set CHESS_REPO_ROOT or move the repo to this Drive path."
        )
    os.chdir(DEFAULT_REPO_ROOT)

print("IN_COLAB:", IN_COLAB)
print("cwd:", Path.cwd().resolve())


In [ ]:
import numpy as np
import os
import sys
import random
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model").exists() and (candidate / "train").exists() and (candidate / "runs").exists():
            return candidate
    raise RuntimeError(f"Cannot find repo root from: {start}")


REPO_ROOT = find_repo_root(Path.cwd())
MODEL_ROOT = REPO_ROOT / "model"
RUNS_ROOT = REPO_ROOT / "runs"
IS_WINDOWS = (os.name == "nt")
IS_COLAB = ("google.colab" in sys.modules)

if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))
    print(f"Added to sys.path: {MODEL_ROOT}")
else:
    print(f"Already in sys.path: {MODEL_ROOT}")

architecture_folder_path = MODEL_ROOT / "architecture_v2"
required_files = ["model.py", "blocks.py", "head.py"]
for file_name in required_files:
    file_path = architecture_folder_path / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required architecture file: {file_path}")

from architecture_v2.model import DGRNChessNetV2 as DGRNChessNet


SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"RUNS_ROOT = {RUNS_ROOT}")
print(f"Global seed = {SEED}")
print(f"Imported model class = {getattr(DGRNChessNet, '__name__', DGRNChessNet)}")
print(f"IS_COLAB = {IS_COLAB}")


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import numpy as np

DATA_ROOT_OVERRIDE = os.environ.get("CHESS_DATA_ROOT", "").strip()
DATA_ROOT = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else (REPO_ROOT / "data" / "process")
STAGE_DATA_LOCAL = IN_COLAB and (os.environ.get("CHESS_STAGE_DATA_LOCAL", "1").strip().lower() not in {"0", "false", "no", "n"})
FORCE_RESTAGE = os.environ.get("CHESS_FORCE_RESTAGE", "0").strip().lower() in {"1", "true", "yes", "y"}
LOCAL_DATA_ROOT = Path(os.environ.get("CHESS_LOCAL_DATA_ROOT", "/content/chess_engine_data/process"))

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Missing data root: {DATA_ROOT}. "
        "Set CHESS_DATA_ROOT or place the shards under REPO_ROOT/data/process."
    )

DATA_ROOT_ACTIVE = DATA_ROOT
if STAGE_DATA_LOCAL:
    if FORCE_RESTAGE and LOCAL_DATA_ROOT.exists():
        shutil.rmtree(LOCAL_DATA_ROOT)
    local_ready = (LOCAL_DATA_ROOT / "train").exists() and (LOCAL_DATA_ROOT / "val").exists() and (LOCAL_DATA_ROOT / "test").exists()
    if not local_ready:
        LOCAL_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
        src = str(DATA_ROOT).rstrip("/\\") + "/"
        dst = str(LOCAL_DATA_ROOT).rstrip("/\\") + "/"
        if shutil.which("rsync"):
            print(f"Staging dataset to local runtime disk via rsync: {DATA_ROOT} -> {LOCAL_DATA_ROOT}")
            subprocess.run(["rsync", "-a", "--delete", src, dst], check=True)
        else:
            print(f"Staging dataset to local runtime disk via shutil: {DATA_ROOT} -> {LOCAL_DATA_ROOT}")
            shutil.copytree(DATA_ROOT, LOCAL_DATA_ROOT, dirs_exist_ok=True)
    DATA_ROOT_ACTIVE = LOCAL_DATA_ROOT

USE_LOCAL_CACHE = False
SPLITS = ("train", "val", "test")


def _sorted_npy_files(split_dir: Path, pattern: str):
    return sorted(split_dir.glob(pattern), key=lambda p: p.name)


def scan_split(split: str):
    split_dir = DATA_ROOT_ACTIVE / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Missing split dir: {split_dir}")

    x_files = _sorted_npy_files(split_dir, "X_*.npy")
    y_files = _sorted_npy_files(split_dir, "y_*.npy")
    if len(x_files) == 0 or len(y_files) == 0:
        raise FileNotFoundError(f"No shards found in {split_dir}. Expect X_*.npy and y_*.npy")

    if len(x_files) != len(y_files):
        raise ValueError(f"Shard count mismatch in {split}: X={len(x_files)} vs y={len(y_files)}")

    shard_sizes = []
    for xf, yf in zip(x_files, y_files):
        X = np.load(xf, mmap_mode="r")
        y = np.load(yf, mmap_mode="r")

        if X.ndim != 4 or X.shape[1:] != (18, 8, 8):
            raise ValueError(f"Bad X shape at {xf.name}: {X.shape} (expect (N,18,8,8))")
        if y.ndim != 1:
            raise ValueError(f"Bad y shape at {yf.name}: {y.shape} (expect (N,))")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"N mismatch at {xf.name} vs {yf.name}: {X.shape[0]} vs {y.shape[0]}")

        if X.dtype != np.uint8:
            print(f"[warn] {split}/{xf.name}: X dtype is {X.dtype}, expect uint8")
        if y.dtype not in (np.float16, np.float32):
            print(f"[warn] {split}/{yf.name}: y dtype is {y.dtype}, expect float16/float32")

        shard_sizes.append(int(X.shape[0]))

    offsets = np.zeros(len(shard_sizes), dtype=np.int64)
    if len(shard_sizes) > 0:
        offsets[1:] = np.cumsum(np.asarray(shard_sizes[:-1], dtype=np.int64))

    meta = {
        "split": split,
        "split_dir": split_dir,
        "x_files": x_files,
        "y_files": y_files,
        "num_shards": len(x_files),
        "shard_sizes": shard_sizes,
        "offsets": offsets,
        "num_samples": int(sum(shard_sizes)),
    }
    return meta


SHARDS = {split: scan_split(split) for split in SPLITS}
for split, meta in SHARDS.items():
    print(
        f"[{split}] root={meta['split_dir']} shards={meta['num_shards']} samples={meta['num_samples']}"
    )
print("DATA_ROOT_SOURCE:", DATA_ROOT)
print("DATA_ROOT_ACTIVE:", DATA_ROOT_ACTIVE)
print("STAGE_DATA_LOCAL:", STAGE_DATA_LOCAL)


In [ ]:
# ===== Cell: Dataset + DataLoader (Colab sanity) =====
import numpy as np
import torch
import random
from torch.utils.data import Dataset, DataLoader


class ShardedNpyDataset(Dataset):
    def __init__(self, meta: dict, dtype_y=torch.float32, use_mmap=True):
        self.meta = meta
        self.x_files = meta["x_files"]
        self.y_files = meta["y_files"]
        self.offsets = np.asarray(meta["offsets"], dtype=np.int64)
        self.num_samples = int(meta["num_samples"])
        self.num_shards = int(meta["num_shards"])
        self.dtype_y = dtype_y
        self.use_mmap = bool(use_mmap)
        self._X = [None] * self.num_shards
        self._y = [None] * self.num_shards

    def __len__(self):
        return self.num_samples

    def _open_shard_if_needed(self, shard_id: int):
        if self._X[shard_id] is None:
            self._X[shard_id] = np.load(self.x_files[shard_id], mmap_mode="r" if self.use_mmap else None)
        if self._y[shard_id] is None:
            self._y[shard_id] = np.load(self.y_files[shard_id], mmap_mode="r" if self.use_mmap else None)

    def __getitem__(self, idx):
        g = int(idx)
        shard_id = int(np.searchsorted(self.offsets, g, side="right") - 1)
        local_i = int(g - self.offsets[shard_id])
        self._open_shard_if_needed(shard_id)
        X = self._X[shard_id][local_i]
        y = self._y[shard_id][local_i]
        X_np = np.array(X, copy=True) if (hasattr(X, "flags") and not X.flags.writeable) else np.asarray(X)
        X = torch.from_numpy(X_np)
        y = torch.as_tensor(y, dtype=self.dtype_y)
        return X, y


class ShardLocalBatchSampler:
    def __init__(self, meta: dict, batch_size: int, drop_last: bool = False, seed: int = 123, shuffle_shards: bool = True, local_shuffle_block: int = 16384, shuffle_within_block: bool = True, shuffle_block_order: bool = True):
        self.offsets = np.asarray(meta["offsets"], dtype=np.int64)
        self.shard_sizes = np.asarray(meta["shard_sizes"], dtype=np.int64)
        self.num_samples = int(meta["num_samples"])
        self.num_shards = int(meta["num_shards"])
        self.batch_size = int(batch_size)
        self.drop_last = bool(drop_last)
        self.seed = int(seed)
        self.shuffle_shards = bool(shuffle_shards)
        self.local_shuffle_block = int(local_shuffle_block)
        self.shuffle_within_block = bool(shuffle_within_block)
        self.shuffle_block_order = bool(shuffle_block_order)
        self.epoch = 0
        self.start_batch = 0

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def set_start_batch(self, start_batch: int):
        self.start_batch = max(0, int(start_batch))

    def __len__(self):
        if self.drop_last:
            return self.num_samples // self.batch_size
        return (self.num_samples + self.batch_size - 1) // self.batch_size

    def _local_order(self, n: int, rng: np.random.Generator):
        idx = np.arange(n, dtype=np.int64)
        block_size = max(1, self.local_shuffle_block)
        if block_size <= 1:
            rng.shuffle(idx)
            return idx
        n_blocks = (n + block_size - 1) // block_size
        blocks = np.arange(n_blocks, dtype=np.int64)
        rng.shuffle(blocks)
        out = np.empty(n, dtype=np.int64)
        pos = 0
        for block_id in blocks:
            start = int(block_id * block_size)
            end = min(n, start + block_size)
            block = idx[start:end].copy()
            if self.shuffle_within_block:
                rng.shuffle(block)
            span = end - start
            out[pos:pos + span] = block
            pos += span
        return out

    def _build_global_blocks(self, rng: np.random.Generator):
        shard_order = np.arange(self.num_shards, dtype=np.int64)
        if self.shuffle_shards:
            rng.shuffle(shard_order)
        block_len = max(self.batch_size, self.local_shuffle_block)
        blocks = []
        for shard_id in shard_order:
            shard_id = int(shard_id)
            n = int(self.shard_sizes[shard_id])
            if n <= 0:
                continue
            start = int(self.offsets[shard_id])
            local_idx = self._local_order(n, rng)
            for local_start in range(0, n, block_len):
                local_end = min(n, local_start + block_len)
                block = local_idx[local_start:local_end]
                if block.size > 0:
                    blocks.append(start + block)
        if self.shuffle_block_order:
            rng.shuffle(blocks)
        return blocks

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        blocks = self._build_global_blocks(rng)
        carry = []
        batch_idx = 0
        for block in blocks:
            block_list = block.tolist()
            i = 0
            n = len(block_list)
            while i < n:
                need = self.batch_size - len(carry)
                j = min(i + need, n)
                carry.extend(block_list[i:j])
                i = j
                if len(carry) == self.batch_size:
                    if batch_idx >= self.start_batch:
                        yield carry
                    batch_idx += 1
                    carry = []
        if (not self.drop_last) and len(carry) > 0:
            if batch_idx >= self.start_batch:
                yield carry


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(ds: Dataset, batch_size: int, shuffle: bool, num_workers: int, device: torch.device, seed: int, persistent_workers: bool = True, batch_sampler=None, prefetch_factor: int = 2):
    pin = (device.type == "cuda")
    generator = torch.Generator()
    generator.manual_seed(seed)
    common = dict(dataset=ds, num_workers=num_workers, pin_memory=pin, persistent_workers=(persistent_workers and num_workers > 0), worker_init_fn=seed_worker if num_workers > 0 else None, generator=generator)
    if num_workers > 0:
        common["prefetch_factor"] = int(prefetch_factor)
    if batch_sampler is not None:
        return DataLoader(batch_sampler=batch_sampler, **common)
    return DataLoader(batch_size=batch_size, shuffle=shuffle, drop_last=False, **common)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

if device.type == "cuda":
    gpu_props = torch.cuda.get_device_properties(0)
    GPU_NAME = gpu_props.name
    GPU_TOTAL_MEM_GB = gpu_props.total_memory / (1024 ** 3)
else:
    GPU_NAME = None
    GPU_TOTAL_MEM_GB = 0.0

print("GPU_NAME:", GPU_NAME)
print("GPU_TOTAL_MEM_GB:", round(GPU_TOTAL_MEM_GB, 2))

train_ds = ShardedNpyDataset(SHARDS["train"], dtype_y=torch.float32, use_mmap=True)
val_ds = ShardedNpyDataset(SHARDS["val"], dtype_y=torch.float32, use_mmap=True)
test_ds = ShardedNpyDataset(SHARDS["test"], dtype_y=torch.float32, use_mmap=True)

if device.type == "cuda":
    if GPU_TOTAL_MEM_GB <= 4.5:
        BATCH_SIZE = 128
        EVAL_BATCH_SIZE = 256
        GRAD_ACCUM_STEPS_DEFAULT = 16
    elif GPU_TOTAL_MEM_GB <= 8.5:
        BATCH_SIZE = 256
        EVAL_BATCH_SIZE = 512
        GRAD_ACCUM_STEPS_DEFAULT = 8
    else:
        BATCH_SIZE = 512
        EVAL_BATCH_SIZE = 1024
        GRAD_ACCUM_STEPS_DEFAULT = 4
else:
    BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    GRAD_ACCUM_STEPS_DEFAULT = 1

NUM_WORKERS = 2 if (device.type == "cuda") else 0
LOCAL_SHUFFLE_BLOCK = 32768
TRAIN_DROP_LAST = True

print("TRAIN_MICRO_BATCH_SIZE:", BATCH_SIZE)
print("EVAL_BATCH_SIZE:", EVAL_BATCH_SIZE)
print("GRAD_ACCUM_STEPS_DEFAULT:", GRAD_ACCUM_STEPS_DEFAULT)
print("NUM_WORKERS:", NUM_WORKERS)

train_batch_sampler = ShardLocalBatchSampler(SHARDS["train"], batch_size=BATCH_SIZE, drop_last=TRAIN_DROP_LAST, seed=SEED + 100, shuffle_shards=True, local_shuffle_block=LOCAL_SHUFFLE_BLOCK, shuffle_within_block=True, shuffle_block_order=True)
train_loader = make_loader(train_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device, seed=SEED + 1, batch_sampler=train_batch_sampler)
val_loader = make_loader(val_ds, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device, seed=SEED + 2)
test_loader = make_loader(test_ds, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device, seed=SEED + 3)

xb, yb = next(iter(train_loader))
print("batch X:", xb.shape, xb.dtype, "batch y:", yb.shape, yb.dtype)
print("X min/max:", float(xb.min()), float(xb.max()), "y min/max:", float(yb.min()), float(yb.max()))
print("train micro-batches/epoch:", len(train_loader), "LOCAL_SHUFFLE_BLOCK:", LOCAL_SHUFFLE_BLOCK, "TRAIN_DROP_LAST:", TRAIN_DROP_LAST)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


In [ ]:
import importlib.util
import os
import sys
import json
from pathlib import Path


def resolve_train_v2_helper_path(repo_root: Path) -> Path:
    candidates = []
    env_dir = os.environ.get("CHESS_TRAIN_V2_DIR", "").strip()
    if env_dir:
        candidates.append(repo_root / env_dir / "ft1_colab_helpers.py")
    candidates.extend([
        repo_root / "train_v2_TF1" / "ft1_colab_helpers.py",
        repo_root / "train_v2_FT1" / "ft1_colab_helpers.py",
    ])
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            return path
    raise FileNotFoundError(f"Cannot find ft1_colab_helpers.py under {repo_root}. Tried: {candidates}")


def load_ft1_helper(repo_root: Path):
    helper_name = "ft1_colab_helpers"
    helper_path = resolve_train_v2_helper_path(repo_root)
    if helper_name in sys.modules:
        del sys.modules[helper_name]
    spec = importlib.util.spec_from_file_location(helper_name, helper_path)
    ft1 = importlib.util.module_from_spec(spec)
    sys.modules[helper_name] = ft1
    assert spec.loader is not None
    spec.loader.exec_module(ft1)
    return ft1, helper_path


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else int(default)


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else float(default)


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return bool(default)
    return raw in {"1", "true", "yes", "y"}

ft1, HELPER_PATH = load_ft1_helper(REPO_ROOT)

GPU_NAME_SAFE = GPU_NAME if "GPU_NAME" in globals() else None
GPU_TOTAL_MEM_GB_SAFE = float(GPU_TOTAL_MEM_GB if "GPU_TOTAL_MEM_GB" in globals() else 0.0)

MODEL_CFG = {
    "num_blocks": 20,
    "hidden_dim": 256,
    "input_channels": 18,
    "drop_path_rate": 0.05,
    "output_mode": "tanh",
}

RUN_NAME = os.environ.get("CHESS_RUN_NAME", "dgrn_5m_ft1_colab_pcgrad_run1")
RUN_DIR = RUNS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESUME_IF_EXISTS = env_bool("CHESS_RESUME_IF_EXISTS", False)
EPOCHS = env_int("CHESS_EPOCHS", 50)
LATEST_CKPT = RUN_DIR / "checkpoints" / "ckpt_latest.pt"
RUN_CONFIG_PATH = RUN_DIR / "reports" / "run_config.json"

saved_train_cfg = {}
if RESUME_IF_EXISTS and LATEST_CKPT.exists() and RUN_CONFIG_PATH.exists():
    saved_run_config = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
    saved_train_cfg = dict(saved_run_config.get("train_cfg", {}))
    PROFILE = {
        "main_batch_size": int(saved_train_cfg.get("main_batch_size", 320)),
        "clean_center_batch_size": int(saved_train_cfg.get("clean_center_batch_size", 40)),
        "ambiguous_center_batch_size": int(saved_train_cfg.get("ambiguous_center_batch_size", 80)),
        "grad_accum_steps": int(saved_train_cfg.get("grad_accum_steps", 2)),
        "eval_batch_size": int(saved_train_cfg.get("eval_batch_size", 2048)),
    }
    print("[resume-config] Reusing saved batch/eval profile from run_config.json")
else:
    BASE_PROFILE = ft1.default_colab_profile(GPU_NAME_SAFE, GPU_TOTAL_MEM_GB_SAFE, os.cpu_count())
    AUTOTUNE_PROFILE = (device.type == "cuda") and (not env_bool("CHESS_DISABLE_PROFILE_AUTOTUNE", False))
    PROFILE = ft1.autotune_ft1_profile(
        model_cfg=MODEL_CFG,
        device=device,
        gpu_name=GPU_NAME_SAFE,
        total_mem_gb=GPU_TOTAL_MEM_GB_SAFE,
        base_profile=BASE_PROFILE,
        use_amp=env_bool("CHESS_USE_AMP", True),
        amp_dtype=os.environ.get("CHESS_AMP_DTYPE", "float16"),
        amp_loss_scale=env_float("CHESS_AMP_LOSS_SCALE", 128.0),
    ) if AUTOTUNE_PROFILE else BASE_PROFILE
    print("[profile] autotune_enabled=", AUTOTUNE_PROFILE)

PROFILE["main_batch_size"] = env_int("CHESS_MAIN_BATCH_SIZE", PROFILE["main_batch_size"])
PROFILE["clean_center_batch_size"] = env_int("CHESS_CLEAN_CENTER_BATCH_SIZE", PROFILE["clean_center_batch_size"])
PROFILE["ambiguous_center_batch_size"] = env_int("CHESS_AMBIGUOUS_CENTER_BATCH_SIZE", PROFILE["ambiguous_center_batch_size"])
PROFILE["grad_accum_steps"] = env_int("CHESS_GRAD_ACCUM_STEPS", PROFILE["grad_accum_steps"])
PROFILE["eval_batch_size"] = env_int("CHESS_EVAL_BATCH_SIZE", PROFILE["eval_batch_size"])

TRAIN_CFG = ft1.FT1TrainConfig(
    run_name=RUN_NAME,
    epochs=EPOCHS,
    main_batch_size=int(PROFILE["main_batch_size"]),
    clean_center_batch_size=int(PROFILE["clean_center_batch_size"]),
    ambiguous_center_batch_size=int(PROFILE["ambiguous_center_batch_size"]),
    grad_accum_steps=int(PROFILE["grad_accum_steps"]),
    eval_batch_size=int(PROFILE["eval_batch_size"]),
    use_amp=env_bool("CHESS_USE_AMP", bool(saved_train_cfg.get("use_amp", True))),
    amp_dtype=os.environ.get("CHESS_AMP_DTYPE", str(saved_train_cfg.get("amp_dtype", "float16"))).strip() or "float16",
    amp_loss_scale=env_float("CHESS_AMP_LOSS_SCALE", float(saved_train_cfg.get("amp_loss_scale", 128.0))),
    preload_shard_dtype=os.environ.get("CHESS_PRELOAD_SHARD_DTYPE", str(saved_train_cfg.get("preload_shard_dtype", "auto"))).strip() or "auto",
    resume_if_exists=RESUME_IF_EXISTS,
    learning_rate=env_float("CHESS_LEARNING_RATE", float(saved_train_cfg.get("learning_rate", 1.0e-4))),
    min_lr=env_float("CHESS_MIN_LR", float(saved_train_cfg.get("min_lr", 1.0e-5))),
    weight_decay=env_float("CHESS_WEIGHT_DECAY", float(saved_train_cfg.get("weight_decay", 1.0e-4))),
    grad_clip_norm=env_float("CHESS_GRAD_CLIP_NORM", float(saved_train_cfg.get("grad_clip_norm", 1.0))),
    val_max_samples=env_int("CHESS_VAL_MAX_SAMPLES", int(saved_train_cfg.get("val_max_samples", 100_000))),
    test_max_samples=env_int("CHESS_TEST_MAX_SAMPLES", int(saved_train_cfg.get("test_max_samples", 200_000))),
    val_num_shards=env_int("CHESS_VAL_NUM_SHARDS", int(saved_train_cfg.get("val_num_shards", 2))),
    test_num_shards=env_int("CHESS_TEST_NUM_SHARDS", int(saved_train_cfg.get("test_num_shards", 4))),
    grad_monitor_every_steps=env_int("CHESS_GRAD_MONITOR_EVERY_STEPS", int(saved_train_cfg.get("grad_monitor_every_steps", 1000))),
    use_backbone_pcgrad=env_bool("CHESS_USE_BACKBONE_PCGRAD", bool(saved_train_cfg.get("use_backbone_pcgrad", True))),
)
GATE_CFG = ft1.FT1GateConfig(
    midband_mae_rel_tol=env_float("CHESS_MIDBAND_MAE_REL_TOL", 0.05),
    stable_slope_abs_tol=env_float("CHESS_STABLE_SLOPE_ABS_TOL", 0.02),
)

print("FT1 helper:", HELPER_PATH)
print("RUN_DIR:", RUN_DIR)
print("DATA_ROOT_ACTIVE:", DATA_ROOT_ACTIVE)
print("GPU_NAME:", GPU_NAME_SAFE)
print("GPU_TOTAL_MEM_GB:", GPU_TOTAL_MEM_GB_SAFE)
print("PROFILE:", PROFILE)
print("TRAIN_CFG:", TRAIN_CFG)
print("GATE_CFG:", GATE_CFG)
print("effective_main_batch:", int(TRAIN_CFG.main_batch_size) * int(TRAIN_CFG.grad_accum_steps))
print("mixed_micro_batch:", int(TRAIN_CFG.main_batch_size) + int(TRAIN_CFG.clean_center_batch_size) + int(TRAIN_CFG.ambiguous_center_batch_size))
print("No Stockfish runtime needed; eval uses cached oracle artifacts already in the repo.")

run_artifacts = ft1.run_ft1_full_retrain(
    model_cfg=MODEL_CFG,
    train_cfg=TRAIN_CFG,
    gate_cfg=GATE_CFG,
    data_root=DATA_ROOT_ACTIVE,
    runs_root=RUNS_ROOT,
    device=device,
    repo_root=REPO_ROOT,
)

print("Run finished.")
print("Run dir:", run_artifacts["run_dir"])
print("Selected checkpoint:", run_artifacts["selected_checkpoint"])
print("Decision summary:", run_artifacts["decision_summary"])


In [ ]:
import importlib.util
import os
import sys
import json
from pathlib import Path


def resolve_train_v2_helper_path(repo_root: Path) -> Path:
    candidates = []
    env_dir = os.environ.get("CHESS_TRAIN_V2_DIR", "").strip()
    if env_dir:
        candidates.append(repo_root / env_dir / "ft1_colab_helpers.py")
    candidates.extend([
        repo_root / "train_v2_TF1" / "ft1_colab_helpers.py",
        repo_root / "train_v2_FT1" / "ft1_colab_helpers.py",
    ])
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            return path
    raise FileNotFoundError(f"Cannot find ft1_colab_helpers.py under {repo_root}. Tried: {candidates}")


def load_ft1_helper(repo_root: Path):
    helper_name = "ft1_colab_helpers"
    helper_path = resolve_train_v2_helper_path(repo_root)
    if helper_name in sys.modules:
        del sys.modules[helper_name]
    spec = importlib.util.spec_from_file_location(helper_name, helper_path)
    ft1 = importlib.util.module_from_spec(spec)
    sys.modules[helper_name] = ft1
    assert spec.loader is not None
    spec.loader.exec_module(ft1)
    return ft1, helper_path


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else int(default)


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else float(default)


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return bool(default)
    return raw in {"1", "true", "yes", "y"}

ft1, HELPER_PATH = load_ft1_helper(REPO_ROOT)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DEFAULT_RUN_NAME = os.environ.get("CHESS_RUN_NAME", "dgrn_5m_ft1_colab_pcgrad_run1")
RUN_DIR = Path(RUN_DIR if "RUN_DIR" in globals() else ft1.resolve_ft1_run_dir(RUNS_ROOT, DEFAULT_RUN_NAME))
hist = ft1.load_history_frame(RUN_DIR)
step_hist = ft1.load_step_history_frame(RUN_DIR)
decision = ft1.load_decision_summary(RUN_DIR)
l4_ref = ft1.load_l4_reference(RUN_DIR)

print("Decision summary:", decision)
display(hist.tail(5))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(hist["epoch"], hist["train_main_objective"], label="train_main_objective")
axes[0, 0].plot(hist["epoch"], hist["train_aux_objective"], label="train_aux_objective")
axes[0, 0].set_title("FT1 Train Objectives")
axes[0, 0].set_xlabel("epoch")
axes[0, 0].set_ylabel("objective")
axes[0, 0].legend()

axes[0, 1].plot(hist["epoch"], hist["oracle_midband_mae_sum_stable"], marker="o", label="FT1 midband MAE")
axes[0, 1].axhline(float(l4_ref["primary"]["oracle_midband_mae_sum_stable"]), color="tab:gray", linestyle="--", label="L4 midband MAE")
ax2 = axes[0, 1].twinx()
ax2.plot(hist["epoch"], hist["oracle_stable_0.7_slope"], marker="s", color="tab:green", label="FT1 slope")
ax2.axhline(float(l4_ref["primary"]["oracle_stable_0.7_slope"]), color="tab:olive", linestyle="--", label="L4 slope")
axes[0, 1].set_title("Hard A-Gate Metrics")
axes[0, 1].set_xlabel("epoch")
axes[0, 1].set_ylabel("oracle_midband_mae_sum_stable")
ax2.set_ylabel("oracle_stable_0.7_slope")
lines1, labels1 = axes[0, 1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[0, 1].legend(lines1 + lines2, labels1 + labels2, loc="best")

axes[1, 0].plot(hist["epoch"], hist["center_score"], marker="o", label="FT1 center_score")
axes[1, 0].axhline(float(l4_ref["center_score"]), color="tab:gray", linestyle="--", label="L4 center_score")
axes[1, 0].set_title("Center Score vs L4")
axes[1, 0].set_xlabel("epoch")
axes[1, 0].set_ylabel("center_score")
axes[1, 0].legend()

if len(step_hist) > 0:
    axes[1, 1].plot(step_hist["global_step"], step_hist["grad_cosine_backbone_pre"], alpha=0.8, label="cosine_pre")
    axes[1, 1].plot(step_hist["global_step"], step_hist["grad_cosine_backbone_post"], alpha=0.8, label="cosine_post")
    axes[1, 1].set_title("Gradient Conflict + BN Sanity")
    axes[1, 1].set_xlabel("global_step")
    axes[1, 1].set_ylabel("gradient cosine")
    lines = []
    labels = []
    l1, lab1 = axes[1, 1].get_legend_handles_labels()
    lines += l1
    labels += lab1
    if "bn_mean_running_var" in hist.columns and "global_step" in hist.columns:
        ax_bn = axes[1, 1].twinx()
        ax_bn.plot(hist["global_step"], hist["bn_mean_running_var"], color="tab:purple", alpha=0.75, label="bn_mean_running_var")
        ax_bn.set_ylabel("bn_mean_running_var")
        l2, lab2 = ax_bn.get_legend_handles_labels()
        lines += l2
        labels += lab2
    axes[1, 1].legend(lines, labels, loc="best")
else:
    axes[1, 1].text(0.5, 0.5, "No step_history rows yet", ha="center", va="center")
    axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import importlib.util
import os
import sys
import json
from pathlib import Path


def resolve_train_v2_helper_path(repo_root: Path) -> Path:
    candidates = []
    env_dir = os.environ.get("CHESS_TRAIN_V2_DIR", "").strip()
    if env_dir:
        candidates.append(repo_root / env_dir / "ft1_colab_helpers.py")
    candidates.extend([
        repo_root / "train_v2_TF1" / "ft1_colab_helpers.py",
        repo_root / "train_v2_FT1" / "ft1_colab_helpers.py",
    ])
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            return path
    raise FileNotFoundError(f"Cannot find ft1_colab_helpers.py under {repo_root}. Tried: {candidates}")


def load_ft1_helper(repo_root: Path):
    helper_name = "ft1_colab_helpers"
    helper_path = resolve_train_v2_helper_path(repo_root)
    if helper_name in sys.modules:
        del sys.modules[helper_name]
    spec = importlib.util.spec_from_file_location(helper_name, helper_path)
    ft1 = importlib.util.module_from_spec(spec)
    sys.modules[helper_name] = ft1
    assert spec.loader is not None
    spec.loader.exec_module(ft1)
    return ft1, helper_path


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else int(default)


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else float(default)


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return bool(default)
    return raw in {"1", "true", "yes", "y"}

ft1, HELPER_PATH = load_ft1_helper(REPO_ROOT)

import torch

DEFAULT_RUN_NAME = os.environ.get("CHESS_RUN_NAME", "dgrn_5m_ft1_colab_pcgrad_run1")
RUN_DIR = Path(RUN_DIR if "RUN_DIR" in globals() else ft1.resolve_ft1_run_dir(RUNS_ROOT, DEFAULT_RUN_NAME))
CKPT_PATH = Path(SELECTED_CHECKPOINT) if "SELECTED_CHECKPOINT" in globals() else ft1.resolve_selected_checkpoint(RUN_DIR, prefer_gate=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, ckpt = ft1.base_lab.load_model_from_checkpoint(CKPT_PATH, device=device)
model.eval()

print(f"Loaded checkpoint: {CKPT_PATH}")
print("helper:", HELPER_PATH)
print("device:", device)
print("epoch:", ckpt.get("epoch"))
print("global_step:", ckpt.get("global_step"))
print("best_any_center_score:", ckpt.get("best_any_center_score"))
print("best_gate_center_score:", ckpt.get("best_gate_center_score"))


In [ ]:
import importlib.util
import os
import sys
import json
from pathlib import Path


def resolve_train_v2_helper_path(repo_root: Path) -> Path:
    candidates = []
    env_dir = os.environ.get("CHESS_TRAIN_V2_DIR", "").strip()
    if env_dir:
        candidates.append(repo_root / env_dir / "ft1_colab_helpers.py")
    candidates.extend([
        repo_root / "train_v2_TF1" / "ft1_colab_helpers.py",
        repo_root / "train_v2_FT1" / "ft1_colab_helpers.py",
    ])
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            return path
    raise FileNotFoundError(f"Cannot find ft1_colab_helpers.py under {repo_root}. Tried: {candidates}")


def load_ft1_helper(repo_root: Path):
    helper_name = "ft1_colab_helpers"
    helper_path = resolve_train_v2_helper_path(repo_root)
    if helper_name in sys.modules:
        del sys.modules[helper_name]
    spec = importlib.util.spec_from_file_location(helper_name, helper_path)
    ft1 = importlib.util.module_from_spec(spec)
    sys.modules[helper_name] = ft1
    assert spec.loader is not None
    spec.loader.exec_module(ft1)
    return ft1, helper_path


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else int(default)


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else float(default)


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return bool(default)
    return raw in {"1", "true", "yes", "y"}

ft1, HELPER_PATH = load_ft1_helper(REPO_ROOT)

import pandas as pd
from IPython.display import display

DEFAULT_RUN_NAME = os.environ.get("CHESS_RUN_NAME", "dgrn_5m_ft1_colab_pcgrad_run1")
RUN_DIR = Path(RUN_DIR if "RUN_DIR" in globals() else ft1.resolve_ft1_run_dir(RUNS_ROOT, DEFAULT_RUN_NAME))
CKPT_PATH = Path(CKPT_PATH if "CKPT_PATH" in globals() else ft1.resolve_selected_checkpoint(RUN_DIR, prefer_gate=True))

selected_eval = ft1.evaluate_saved_checkpoint(
    checkpoint_path=CKPT_PATH,
    data_root=DATA_ROOT_ACTIVE,
    pooled_center_bundle_dir=REPO_ROOT / "experiments" / "failure_b_resolution_suite" / "outputs" / "cache" / "pooled_center_bundle",
    oracle_role_bundle_dir=REPO_ROOT / "experiments" / "oc2_joint_oracle_full_model_pilot" / "outputs" / "cache" / "oracle_role_bundle",
    device=device,
    eval_batch_size=TRAIN_CFG.eval_batch_size if "TRAIN_CFG" in globals() else 1024,
    test_max_samples=TRAIN_CFG.test_max_samples if "TRAIN_CFG" in globals() else 200_000,
    test_num_shards=TRAIN_CFG.test_num_shards if "TRAIN_CFG" in globals() else 4,
)
l4_ref = ft1.load_l4_reference(RUN_DIR)

keys = [
    "oracle_midband_mae_sum_stable",
    "oracle_stable_0.7_slope",
    "oracle_center_amp_ratio",
    "oracle_center_false_0.1eq",
    "oracle_center_false_0.2eq",
    "pooled_center_mae",
    "pooled_center_amp_ratio",
    "pooled_center_false_0.1eq",
    "pooled_center_false_0.2eq",
    "center_score",
    "clean_center_mae",
    "clean_center_amp_ratio",
    "clean_center_false_0.1eq",
    "ambiguous_center_mae",
    "ambiguous_center_amp_ratio",
    "ambiguous_center_false_0.1eq",
]

compare = pd.DataFrame([
    {"label": "L4_reference", **{k: l4_ref.get(k, l4_ref.get("primary", {}).get(k)) for k in keys}},
    {"label": "FT1_selected", **{k: selected_eval.get(k) for k in keys}},
])
display(compare.round(6))


In [ ]:
import importlib.util
import os
import sys
import json
from pathlib import Path


def resolve_train_v2_helper_path(repo_root: Path) -> Path:
    candidates = []
    env_dir = os.environ.get("CHESS_TRAIN_V2_DIR", "").strip()
    if env_dir:
        candidates.append(repo_root / env_dir / "ft1_colab_helpers.py")
    candidates.extend([
        repo_root / "train_v2_TF1" / "ft1_colab_helpers.py",
        repo_root / "train_v2_FT1" / "ft1_colab_helpers.py",
    ])
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            return path
    raise FileNotFoundError(f"Cannot find ft1_colab_helpers.py under {repo_root}. Tried: {candidates}")


def load_ft1_helper(repo_root: Path):
    helper_name = "ft1_colab_helpers"
    helper_path = resolve_train_v2_helper_path(repo_root)
    if helper_name in sys.modules:
        del sys.modules[helper_name]
    spec = importlib.util.spec_from_file_location(helper_name, helper_path)
    ft1 = importlib.util.module_from_spec(spec)
    sys.modules[helper_name] = ft1
    assert spec.loader is not None
    spec.loader.exec_module(ft1)
    return ft1, helper_path


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else int(default)


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else float(default)


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return bool(default)
    return raw in {"1", "true", "yes", "y"}

ft1, HELPER_PATH = load_ft1_helper(REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

DEFAULT_RUN_NAME = os.environ.get("CHESS_RUN_NAME", "dgrn_5m_ft1_colab_pcgrad_run1")
RUN_DIR = Path(RUN_DIR if "RUN_DIR" in globals() else ft1.resolve_ft1_run_dir(RUNS_ROOT, DEFAULT_RUN_NAME))
CKPT_PATH = Path(CKPT_PATH if "CKPT_PATH" in globals() else ft1.resolve_selected_checkpoint(RUN_DIR, prefer_gate=True))

if "selected_eval" not in globals():
    selected_eval = ft1.evaluate_saved_checkpoint(
        checkpoint_path=CKPT_PATH,
        data_root=DATA_ROOT_ACTIVE,
        pooled_center_bundle_dir=REPO_ROOT / "experiments" / "failure_b_resolution_suite" / "outputs" / "cache" / "pooled_center_bundle",
        oracle_role_bundle_dir=REPO_ROOT / "experiments" / "oc2_joint_oracle_full_model_pilot" / "outputs" / "cache" / "oracle_role_bundle",
        device=device,
        eval_batch_size=TRAIN_CFG.eval_batch_size if "TRAIN_CFG" in globals() else 1024,
        test_max_samples=TRAIN_CFG.test_max_samples if "TRAIN_CFG" in globals() else 200_000,
        test_num_shards=TRAIN_CFG.test_num_shards if "TRAIN_CFG" in globals() else 4,
    )
role_metrics = selected_eval["role_eval"]["metrics"]
role_df = pd.DataFrame([
    {"role": "clean_center", **role_metrics["clean_center"]},
    {"role": "center_ambiguous", **role_metrics["center_ambiguous"]},
])
display(role_df.round(6))

pred = np.asarray(selected_eval["role_eval"]["pred"], dtype=np.float64)
oracle = np.asarray(selected_eval["role_eval"]["oracle"], dtype=np.float64)
role_code = np.asarray(selected_eval["role_eval"]["role_code"], dtype=np.int64)
role_names = np.where(role_code == 0, "clean_center", "center_ambiguous")

fig, ax = plt.subplots(figsize=(8, 8))
for role in ["clean_center", "center_ambiguous"]:
    mask = (role_names == role)
    ax.scatter(oracle[mask], pred[mask], s=20, alpha=0.65, label=role)
ax.axline((0, 0), slope=1.0, color="black", linestyle="--", linewidth=1.0)
ax.set_title("FT1 Role-Specific Oracle vs Prediction")
ax.set_xlabel("oracle")
ax.set_ylabel("prediction")
ax.legend()
plt.show()
